# Río Mana — SegmentEveryGrain, procesamiento completo por teselas

Versión **experimental para Colab**. Basada en el cuaderno de Zoltán Sylvester y el piloto del río Mana. JPEG de ~1 mm/píxel, worldfile y **EPSG:32619**. Conserva la resolución original. Produce resultados por tesela y un GeoPackage final. **Primero ejecutar `MAX_TESELAS=2` y comprobar las salidas.**

Los objetos se asignan a una sola tesela por su centroide en la zona interior (core). Esto evita duplicados habituales, pero los clastos cortados en los bordes de las ventanas **no se reconstruyen**: se marcan como posibles truncamientos. La segmentación y las medidas requieren validación. Las salidas intermedias se guardan en Drive para reanudar.

## 1. Instalación (GPU T4 recomendada)

In [ ]:
import sys
!{sys.executable} -m pip -q install segmenteverygrain sam2 rtree geopandas pyogrio affine
!wget -q -nc https://huggingface.co/facebook/sam2.1-hiera-large/resolve/main/sam2.1_hiera_large.pt
!wget -q -nc https://raw.githubusercontent.com/zsylvester/segmenteverygrain/main/models/seg_model_smooth_labels.pt


## 2. Modelos preentrenados

In [ ]:
import torch, numpy as np, pandas as pd, matplotlib.pyplot as plt
import segmenteverygrain as seg
import segmenteverygrain.interactions as si
from sam2.build_sam import build_sam2
from pathlib import Path
from PIL import Image
from affine import Affine
from shapely.geometry import shape
from shapely.ops import transform as shapely_transform
import geopandas as gpd
import json, gc, time, traceback

assert torch.cuda.is_available(), 'Activar GPU en Colab antes de continuar'
sam = build_sam2('configs/sam2.1/sam2.1_hiera_l.yaml',
                 'sam2.1_hiera_large.pt', device='cuda')
unet = seg.load_unet('seg_model_smooth_labels.pt')
print('GPU:', torch.cuda.get_device_name(0))

## 3. Configuración del ortomosaico y salida

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
CARPETA = Path('/content/drive/MyDrive/gravas_mana')  # AJUSTAR
JPG = CARPETA / 'odm_orthophoto.jpg'               # AJUSTAR
WLD = CARPETA / 'odm_orthophoto_jpg.wld'           # AJUSTAR
SALIDA = CARPETA / 'resultados_completo'
SALIDA.mkdir(parents=True, exist_ok=True)
EPSG = 'EPSG:32619'
TAM = 1024
SOLAPE = 128
MIN_AREA = 100.0       # Cambiar a 400.0 si se desea replicar piloto original
DBS_MAX_DIST = 20.0
MAX_TESELAS = 2        # PRUEBA; poner None para ejecutar TODO
REANUDAR = True
GUARDAR_PNG = True
assert JPG.exists() and WLD.exists(), 'Revisar JPG y WLD'
A,D,B,E,C,F = map(float, WLD.read_text().split())
transform = Affine(A,B,C-(A+B)/2,D,E,F-(D+E)/2)
px_per_m = 1 / ((abs(A)+abs(E))/2)
Image.MAX_IMAGE_PIXELS = None
with Image.open(JPG) as im:
    ANCHO, ALTO = im.size
print('Dimensiones:',ANCHO,ALTO,'px; escala:',px_per_m,'px/m')
print('CRS:',EPSG, 'Transformación:', transform)

## 4. Retícula con solape y zona de responsabilidad única

Las teselas tienen paso `TAM - SOLAPE`. La zona interior asignada a cada tesela se separa por la mitad del solape con la tesela vecina. Se conservan solo los granos cuyo centroide cae en esa zona. Los granos que tocan el borde físico de su ventana se señalan en la tabla.

In [ ]:
def inicios(n, tam, paso):
    valores = list(range(0, n, paso))
    return [v for v in valores if v < n]

PASO = TAM - SOLAPE
xs, ys = inicios(ANCHO,TAM,PASO), inicios(ALTO,TAM,PASO)

def ventanas():
    for iy,y in enumerate(ys):
        for ix,x in enumerate(xs):
            w,h = min(TAM,ANCHO-x), min(TAM,ALTO-y)
            izq = 0 if ix == 0 else SOLAPE/2
            sup = 0 if iy == 0 else SOLAPE/2
            der = w if ix == len(xs)-1 else min(w, PASO+SOLAPE/2)
            inf = h if iy == len(ys)-1 else min(h, PASO+SOLAPE/2)
            yield dict(id=f'x{x:05d}_y{y:05d}', x=x,y=y,w=w,h=h,
                       core=(izq,sup,der,inf))

plan = list(ventanas())
print('Teselas previstas:',len(plan),'(',len(xs),'columnas ×',len(ys),'filas )')
print('Primeras:',plan[:2])

## 5. Procesar teselas, exportar máscaras vistosas y mediciones

Se guardan por tesela: PNG de segmentación, CSV de medidas, GeoJSON en UTM y un archivo `ok` al terminar. La exportación de figuras utiliza el `fig` devuelto por `sam_segmentation`, no el editor interactivo. Se emplean las funciones originales de SegmentEveryGrain para medir y resumir.

In [ ]:
def procesar_tesela(v):
    tid=v['id']; carpeta=SALIDA/tid
    carpeta.mkdir(exist_ok=True)
    ok=carpeta/'completado.ok'
    if REANUDAR and ok.exists():
        print('OMITIDA (completada):',tid)
        return 'omitida'
    inicio=time.perf_counter()
    with Image.open(JPG) as im:
        image=np.asarray(im.crop((v['x'],v['y'],v['x']+v['w'],v['y']+v['h'])).convert('RGB'))
    image_pred=seg.predict_image(image,unet,I=256)
    labels,coords=seg.label_grains(image,image_pred,dbs_max_dist=DBS_MAX_DIST)
    all_grains,labels,mask_all,grain_data,fig,ax=seg.sam_segmentation(
        sam,image,image_pred,coords,labels,min_area=MIN_AREA,
        plot_image=True,remove_edge_grains=False,remove_large_objects=False)
    if GUARDAR_PNG and fig is not None:
        fig.savefig(carpeta/'segmentacion.png',dpi=180,bbox_inches='tight')
    if fig is not None: plt.close(fig)
    grains=si.polygons_to_grains(all_grains,image=image)
    for g in grains: g.measure()
    # Exportaciones nativas: conservar medidas de SEG sin redefinir ejes.
    si.save_grains(str(carpeta/'granos_local.geojson'),grains)
    resumen=si.save_summary(str(carpeta/'resumen_original.csv'),grains,px_per_m=px_per_m)
    if not isinstance(resumen,pd.DataFrame):
        resumen=pd.read_csv(carpeta/'resumen_original.csv')
    geo=json.loads((carpeta/'granos_local.geojson').read_text())
    feats=geo.get('features',[])
    if len(feats)!=len(resumen):
        raise ValueError(f'{tid}: {len(feats)} geometrías y {len(resumen)} filas: revisar correspondencia antes de unir')
    registros=[]; geometrias=[]
    l,t,r,b=v['core']
    for i,feature in enumerate(feats):
        if not feature.get('geometry'): continue
        p=shape(feature['geometry'])
        cx,cy=p.centroid.x,p.centroid.y
        if not (l<=cx<r and t<=cy<b): continue
        # Las coordenadas de las geometrías originales son píxeles locales.
        geom=shapely_transform(lambda xx,yy,z=None: transform*(xx+v['x'],yy+v['y']),p)
        fila=resumen.iloc[i].to_dict()
        fila.update(id_grano=f'{tid}_g{i:05d}',tesela=tid,
                    pixel_col=v['x']+cx,pixel_fila=v['y']+cy,
                    x_utm=geom.centroid.x,y_utm=geom.centroid.y,
                    toca_borde=(p.bounds[0]<=1 or p.bounds[1]<=1 or
                                p.bounds[2]>=v['w']-1 or p.bounds[3]>=v['h']-1))
        registros.append(fila); geometrias.append(geom)
    gdf=gpd.GeoDataFrame(registros,geometry=geometrias,crs=EPSG)
    # GeoJSON por tesela para poder consolidar sin volver a segmentar.
    gdf.to_file(carpeta/'granos_utm.geojson',driver='GeoJSON')
    gdf.drop(columns='geometry').to_csv(carpeta/'mediciones.csv',index=False)
    duracion=time.perf_counter()-inicio
    (carpeta/'registro.json').write_text(json.dumps({
        'tesela':tid,'semillas':len(coords),'mascaras':len(feats),
        'granos_conservados':len(gdf),'segundos':duracion,
        'min_area':MIN_AREA,'dbs_max_dist':DBS_MAX_DIST,
        'toca_borde':int(gdf.toca_borde.sum()) if len(gdf) else 0},indent=2))
    ok.write_text('OK')
    print(tid,'semillas',len(coords),'máscaras',len(feats),
          'conservados',len(gdf),'tiempo',round(duracion,1),'s')
    del image,image_pred,labels,mask_all,grain_data,grains,all_grains,gdf
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return 'completada'

pendientes=[v for v in plan if not (REANUDAR and (SALIDA/v['id']/'completado.ok').exists())]
if MAX_TESELAS is not None: pendientes=pendientes[:MAX_TESELAS]
print('A ejecutar en esta sesión:',len(pendientes))
for v in pendientes:
    try:
        procesar_tesela(v)
    except Exception as e:
        print('ERROR en',v['id'],repr(e))
        (SALIDA/'errores.log').open('a').write(v['id']+' '+repr(e)+'\n')
        traceback.print_exc()
        # Se detiene para corregir problemas estructurales antes de seguir.
        raise

## 6. Consolidar en GeoPackage, CSV y tabla de control

Ejecutar después de completar las teselas deseadas. Se incluyen solo teselas con marcador `completado.ok`. La primera prueba con dos teselas produce un conjunto **parcial**, no la cartografía total.

In [ ]:
rutas=[SALIDA/v['id']/'granos_utm.geojson' for v in plan
       if (SALIDA/v['id']/'completado.ok').exists()]
registros=[json.loads((p.parent/'registro.json').read_text()) for p in rutas]
pd.DataFrame(registros).to_csv(SALIDA/'control_teselas.csv',index=False)
print('Teselas completas:',len(rutas),'de',len(plan))
if rutas:
    partes=[gpd.read_file(p) for p in rutas]
    unidos=gpd.GeoDataFrame(pd.concat(partes,ignore_index=True),crs=EPSG)
    unidos.to_file(SALIDA/'granos_mana.gpkg',layer='granos',driver='GPKG')
    unidos.drop(columns='geometry').to_csv(SALIDA/'granos_mana.csv',index=False)
    print('Clastos exportados:',len(unidos))
    print('GPKG:',SALIDA/'granos_mana.gpkg')
else:
    print('No hay teselas terminadas')

## 7. Comprobaciones antes de lanzar la imagen completa

1. Abrir `resultados_completo/x00000_y00000/segmentacion.png` y `granos_utm.geojson` en QGIS.
2. Comprobar que las coordenadas corresponden al río Mana, y que `mediciones.csv` coincide con las geometrías.
3. Verificar el porcentaje de granos con `toca_borde=True`; son posibles truncamientos.
4. Si la prueba funciona, establecer `MAX_TESELAS=None` y volver a ejecutar la celda 5. Las teselas completadas se omitirán.
5. Ejecutar la consolidación al finalizar. La deduplicación por centroide **no fusiona** fragmentos de un mismo clasto entre teselas; esta limitación debe considerarse en el análisis.

**Nota de rendimiento:** el JPEG grande puede requerir decodificación repetida. Si resulta lento, conviene convertirlo previamente a GeoTIFF en bloques y leer ventanas con rasterio. El coste de SAM también puede ser elevado; no se presupone que todo el mosaico termine en una sola sesión.
